In [ ]:
!pip install google-generativeai langchain langchain-core

In [7]:
import google.generativeai as genai
from getpass import getpass
from langchain.prompts import ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser
from langchain_core.runnables import RunnableSequence, RunnableParallel, RunnableLambda

In [3]:
api_key = getpass("🔑 Enter your Gemini API key: ")
genai.configure(api_key=api_key)

# Create a custom LLM class that wraps Gemini for LangChain compatibility
from langchain_core.language_models.llms import LLM
from typing import Optional, List, Mapping, Any

class GeminiLLM(LLM):
    model_name: str = "gemini-2.5-flash"
    
    @property
    def _llm_type(self) -> str:
        return "gemini"
    
    def _call(self, prompt: str, stop: Optional[List[str]] = None, **kwargs) -> str:
        model = genai.GenerativeModel(self.model_name)
        response = model.generate_content(prompt)
        return response.text
    
    @property
    def _identifying_params(self) -> Mapping[str, Any]:
        return {"model_name": self.model_name}

model = GeminiLLM()

## Define Base Components

In [13]:
# Prompt template
prompt = ChatPromptTemplate.from_template("Write a short poem about {topic}.")

# Output parser
parser = StrOutputParser()

# FIX: Use a regular function instead of RunnableLambda
base_seq = prompt | (lambda x: x.to_string()) | model | parser

## Add Custom Logic with RunnableLambda

In [11]:
# Prompt template
prompt = ChatPromptTemplate.from_template("Write a short poem about {topic}.")

# Output parser
parser = StrOutputParser()


base_seq = prompt | (lambda x: x.to_string()) | model | parser

In [14]:
result = base_seq.invoke({"topic": "autumn leaves"})
print(result)

Crimson, gold, and fiery red,
A painted canopy overhead.
They dance and twirl on autumn's breeze,
Then softly fall from weary trees.

A rustling carpet, crisp and deep,
Where secrets of the season sleep.


## Use runnable lambda directly

In [15]:
def poem_func(inputs: dict) -> str:
    topic = inputs["topic"].title()
    prompt_input = {"topic": topic}
    poem = base_seq.invoke(prompt_input)
    return poem + "!"

# Wrap in RunnableLambda
poem_chain = RunnableLambda(poem_func)

result = poem_chain.invoke({"topic": "mountains"})
print(result)

Silent giants, old and grand,
Rising proudly from the land.
Their rocky peaks, where eagles roam,
A majestic, silent home.!
